# Optunaによるパラメータのオートチューニング

In [1]:
!pip install -qq optuna kaggle-environments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 7.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 65.8 MB

In [2]:
"""Colabで実行するパラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 150
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """平均得点を返す。"""

    #牛乳を通常売却する最低価
    strategy.StrategyConfig.MILK_SELL_PRICE = trial.suggest_int(
        "MILK_SELL_PRICE",
        120,
        240,
        step=10,
    )

    #終盤以外で一度に売却する牛乳の上限数
    strategy.StrategyConfig.MILK_SELL_BATCH = trial.suggest_int(
        "MILK_SELL_BATCH",
        1,
        12,
    )

    #牛乳の緊急売却を開始する非種子アイテムの保有数
    strategy.StrategyConfig.SHED_EMERGENCY_SELL_LEVEL = trial.suggest_int(
        "SHED_EMERGENCY_SELL_LEVEL",
        75,
        99,
        step=2,
    )

    #牛乳・メロンの強制売却を開始するSTEP
    strategy.StrategyConfig.FINAL_SELL_STEP = trial.suggest_int(
        "FINAL_SELL_STEP",
        690,
        715,
        step=5,
    )


    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-10 23:11:36,790] A new study created in memory with name: no-name-268fb12e-ff0f-476b-bf0b-c39d2fc61f79


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-09-10 23:14:48,081] Trial 0 finished with value: 68988.05 and parameters: {'MILK_SELL_PRICE': 160, 'MILK_SELL_BATCH': 12, 'SHED_EMERGENCY_SELL_LEVEL': 93, 'FINAL_SELL_STEP': 705}. Best is trial 0 with value: 68988.05.
[I 2026-09-10 23:17:58,155] Trial 1 finished with value: 68115.35 and parameters: {'MILK_SELL_PRICE': 140, 'MILK_SELL_BATCH': 2, 'SHED_EMERGENCY_SELL_LEVEL': 75, 'FINAL_SELL_STEP': 715}. Best is trial 0 with value: 68988.05.
[I 2026-09-10 23:21:10,956] Trial 2 finished with value: 68661.55 and parameters: {'MILK_SELL_PRICE': 190, 'MILK_SELL_BATCH': 9, 'SHED_EMERGENCY_SELL_LEVEL': 75, 'FINAL_SELL_STEP': 715}. Best is trial 0 with value: 68988.05.
[I 2026-09-10 23:24:22,703] Trial 3 finished with value: 71325.75 and parameters: {'MILK_SELL_PRICE': 220, 'MILK_SELL_BATCH': 3, 'SHED_EMERGENCY_SELL_LEVEL': 79, 'FINAL_SELL_STEP': 695}. Best is trial 3 with value: 71325.75.
[I 2026-09-10 23:27:34,392] Trial 4 finished with value: 68997.775 and parameters: {'MILK_SELL_PRIC